# Análise dos resultados de segmentação e detecção dos óstios

## Objetivo

Este notebook realiza uma análise exploratória dos resultados da segmentação dos óstios a partir do pipeline. São apresentados gráficos e estatísticas para avaliar a qualidade das segmentações.

Resume status dos óstios, distâncias e Dice dos resultados canônicos de treino, validação e teste.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from typing import Any, Dict, Optional, Sequence
from utils.project.results.schema import normalize_result_status, result_status_label_pt
from utils.comparison_utils.segmentation_eda import (
    _get_split_df,
    _has_success_status_column,
    _normalized_success_statuses,
    _success_status_series,
)
from pathlib import Path
import seaborn as sns


def _get_split_title(split_name: str) -> str:
    return str(split_name).upper()


def _get_status_color(split_name: str) -> str:
    return {
        "train": "skyblue",
        "val": "lightgreen",
        "test": "salmon",
    }.get(split_name, "steelblue")


def plot_status_distribution_by_subset(
    data_by_resolution: Optional[Dict[str, Any]],
    split_name: str,
    status_column: str = "status",
) -> None:
    """Plota a distribuição de status para High e Mid dentro de um subset."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    title_suffix = _get_split_title(split_name)
    color = _get_status_color(split_name)

    for idx, resolution in enumerate(["high", "mid"]):
        df_split = _get_split_df(data_by_resolution, resolution, split_name)
        ax = axes[idx]

        if df_split is None or df_split.empty or status_column not in df_split.columns:
            ax.text(
                0.5,
                0.5,
                f"Sem dados para {resolution.upper()}",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_title(
                f"{resolution.upper()} Resolution - Distribuição dos Status ({title_suffix})"
            )
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        status_counts = (
            df_split[status_column]
            .map(normalize_result_status)
            .map(result_status_label_pt)
            .value_counts()
        )
        status_counts.plot(kind="bar", color=color, edgecolor="black", ax=ax)
        # Ajusta o limite superior para evitar que os rótulos sobreponham o título
        max_count = max(status_counts.values) if len(status_counts.values) > 0 else 0
        ax.set_ylim(0, max_count * 1.18 if max_count > 0 else 1)
        ax.set_title(
            f"{resolution.upper()} Resolution - Distribuição dos Status ({title_suffix})",
            pad=12,
        )
        ax.set_xlabel("Status")
        ax.set_ylabel("Número de Imagens")
        ax.tick_params(axis="x", rotation=30)
        ax.grid(axis="y", linestyle="--", alpha=0.7)

        for i, value in enumerate(status_counts.values):
            text_offset = max_count * 0.03 if max_count > 0 else 0.1
            ax.text(
                i,
                value + text_offset,
                str(value),
                ha="center",
                va="bottom",
                fontweight="bold",
            )

    plt.tight_layout()
    plt.show()


def plot_success_error_by_subset(
    data_by_resolution: Optional[Dict[str, Any]],
    split_name: str,
    success_status: Sequence[str],
    status_column: str = "status",
    success_color: str = "#4CAF50",
    error_color: str = "#F44336",
) -> None:
    """Plota acertos vs erros para High e Mid dentro de um subset."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    title_suffix = _get_split_title(split_name)

    for idx, resolution in enumerate(["high", "mid"]):
        df_split = _get_split_df(data_by_resolution, resolution, split_name)
        ax = axes[idx]

        if (
            df_split is None
            or df_split.empty
            or not _has_success_status_column(df_split, status_column)
        ):
            ax.text(
                0.5,
                0.5,
                f"Sem dados para {resolution.upper()}",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
            ax.set_title(
                f"{resolution.upper()} Resolution - Acertos vs Erros ({title_suffix})"
            )
            ax.set_xticks([])
            ax.set_yticks([])
            continue

        total = len(df_split)
        normalized_status = _success_status_series(df_split, status_column)
        normalized_success = _normalized_success_statuses(success_status)
        successful_count = int(normalized_status.isin(normalized_success).sum())
        error_count = total - successful_count

        success_pct = 100 * successful_count / total if total > 0 else 0
        error_pct = 100 * error_count / total if total > 0 else 0

        bars = ax.bar(
            ["Acertos", "Erros"],
            [successful_count, error_count],
            color=[success_color, error_color],
            alpha=0.8,
            edgecolor="black",
        )

        ax.set_ylabel("Número de Imagens")
        # Ajusta limite superior do eixo y para dar espaço aos rótulos
        max_count = max(successful_count, error_count)
        ax.set_ylim(0, max_count * 1.18 if max_count > 0 else 1)
        ax.set_title(
            f"{resolution.upper()} Resolution - Acertos vs Erros ({title_suffix})",
            pad=12,
        )
        ax.grid(axis="y", alpha=0.3)

        for i, (bar, value) in enumerate(zip(bars, [successful_count, error_count])):
            percentage = [success_pct, error_pct][i]
            height = bar.get_height()
            text_offset = max_count * 0.03 if max_count > 0 else 0.1
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + text_offset,
                f"{int(value)}\n({percentage:.1f}%)",
                ha="center",
                va="bottom",
                fontweight="bold",
            )

    plt.tight_layout()
    plt.show()


def plot_distance_distribution_by_subset(
    data_by_resolution: Optional[Dict[str, Any]],
    split_name: str,
    left_column: str = "left_dist_mm",
    right_column: str = "right_dist_mm",
    tolerance_mm: float = 7.0,
    bins: Optional[int] = None,
    high_color: str = "#FF6B6B",
    mid_color: str = "#4ECDC4",
) -> None:
    """Plota histogramas de distâncias left/right para High e Mid dentro de um subset."""
    if bins is None:
        bins = {"train": 25, "val": 15, "test": 25}.get(split_name, 25)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    title_suffix = _get_split_title(split_name)

    for idx, resolution in enumerate(["high", "mid"]):
        df_split = _get_split_df(data_by_resolution, resolution, split_name)
        ax_left = axes[0, idx]
        ax_right = axes[1, idx]

        if df_split is None or df_split.empty:
            for ax in [ax_left, ax_right]:
                ax.text(
                    0.5,
                    0.5,
                    f"Sem dados para {resolution.upper()}",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                ax.set_xticks([])
                ax.set_yticks([])
            ax_left.set_title(
                f"{resolution.upper()} - Left Distance Distribution ({title_suffix})"
            )
            ax_right.set_title(
                f"{resolution.upper()} - Right Distance Distribution ({title_suffix})"
            )
            continue

        left_dist = (
            pd.to_numeric(df_split[left_column], errors="coerce").dropna()
            if left_column in df_split.columns
            else pd.Series(dtype=float)
        )
        right_dist = (
            pd.to_numeric(df_split[right_column], errors="coerce").dropna()
            if right_column in df_split.columns
            else pd.Series(dtype=float)
        )

        # Remove valores infinitos que causam erro em np.histogram
        if not left_dist.empty:
            left_dist = left_dist[np.isfinite(left_dist)]
        if not right_dist.empty:
            right_dist = right_dist[np.isfinite(right_dist)]
        color = high_color if resolution == "high" else mid_color

        # Contagem de valores abaixo e acima do limiar
        left_below = int((left_dist <= tolerance_mm).sum())
        left_above = int((left_dist > tolerance_mm).sum())
        right_below = int((right_dist <= tolerance_mm).sum())
        right_above = int((right_dist > tolerance_mm).sum())

        print(f"\n{resolution.upper()} - {title_suffix}")
        print(
            f"  Left Distance: {left_below} abaixo de {tolerance_mm}mm, {left_above} acima"
        )
        print(
            f"  Right Distance: {right_below} abaixo de {tolerance_mm}mm, {right_above} acima"
        )

        ax_left.hist(left_dist, bins=bins, alpha=0.8, color=color, edgecolor="black")
        ax_left.axvline(
            tolerance_mm,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Tolerância ({tolerance_mm:g}mm)",
        )
        ax_left.set_xlabel("Left Distance (mm)")
        ax_left.set_ylabel("Frequência")
        ax_left.set_title(
            f"{resolution.upper()} - Left Distance Distribution ({title_suffix})"
        )
        ax_left.legend()
        ax_left.grid(axis="y", alpha=0.3)

        ax_right.hist(right_dist, bins=bins, alpha=0.8, color=color, edgecolor="black")
        ax_right.axvline(
            tolerance_mm,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Tolerância ({tolerance_mm:g}mm)",
        )
        ax_right.set_xlabel("Right Distance (mm)")
        ax_right.set_ylabel("Frequência")
        ax_right.set_title(
            f"{resolution.upper()} - Right Distance Distribution ({title_suffix})"
        )
        ax_right.legend()
        ax_right.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_dice_distribution_for_publication(
    dice_values: pd.Series,
    *,
    output_path: str | Path | None = None,
    dpi: int = 300,
    color: str = "skyblue",
    bins: int = 20,
) -> tuple[Any, dict[str, float | int]]:
    """Plota Dice, média e faixa de um desvio padrão para publicação."""
    values = pd.to_numeric(dice_values, errors="coerce").dropna()
    if values.empty:
        raise ValueError("dice_values não contém valores numéricos válidos.")

    mean = float(values.mean())
    std = float(values.std())
    fig, ax = plt.subplots(figsize=(7.2, 5.0), dpi=dpi)
    sns.histplot(
        values,
        bins=bins,
        kde=True,
        color=color,
        edgecolor="black",
        ax=ax,
    )
    ax.axvline(mean, color="red", linewidth=2, label=f"Média = {mean:.3f}")
    ax.axvline(
        mean - std,
        color="orange",
        linestyle="--",
        linewidth=2,
        label=f"-1σ = {mean - std:.3f}",
    )
    ax.axvline(
        mean + std,
        color="orange",
        linestyle="--",
        linewidth=2,
        label=f"+1σ = {mean + std:.3f}",
    )
    ax.axvspan(mean - std, mean + std, color="orange", alpha=0.15)
    ax.set_xlabel("Dice Score", fontsize=16)
    ax.set_ylabel("Frequência", fontsize=16)
    ax.tick_params(axis="both", labelsize=13)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=12)
    fig.tight_layout()

    if output_path is not None:
        destination = Path(output_path)
        destination.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(destination, dpi=dpi, bbox_inches="tight")

    return fig, {"count": int(values.size), "mean": mean, "std": std}


def plot_dice_distribution_by_subset(
    df_mid: Optional[pd.DataFrame], df_high: Optional[pd.DataFrame], subset_label: str
) -> None:
    """Plota distribuição dos Dice scores para mid e high por subconjunto."""
    # Mostra histogramas de Dice para Mid e High no mesmo painel.
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    if df_mid is not None and not df_mid.empty:
        # Converte e remove valores inválidos de Dice.
        dice_scores_mid = pd.to_numeric(
            df_mid["dice_artery"],
            errors="coerce",
        ).dropna()
        mu_mid = dice_scores_mid.mean()
        sigma_mid = dice_scores_mid.std()
        # KDE mostra a forma da distribuição.
        sns.histplot(
            dice_scores_mid,
            bins=20,
            kde=True,
            color="skyblue",
            edgecolor="black",
            ax=axes[0],
        )
        axes[0].axvline(
            mu_mid,
            color="red",
            linestyle="-",
            linewidth=2,
            label=f"Media = {mu_mid:.3f}",
        )
        axes[0].axvline(
            mu_mid - sigma_mid,
            color="orange",
            linestyle="--",
            linewidth=2,
            label=f"-1sigma = {mu_mid - sigma_mid:.3f}",
        )
        axes[0].axvline(
            mu_mid + sigma_mid,
            color="orange",
            linestyle="--",
            linewidth=2,
            label=f"+1sigma = {mu_mid + sigma_mid:.3f}",
        )
        axes[0].axvspan(
            mu_mid - sigma_mid, mu_mid + sigma_mid, color="orange", alpha=0.15
        )
        # Marca média e faixa de 1 sigma.
        axes[0].set_title(f"Distribuicao dos Dice Scores - Mid Res ({subset_label})")
        axes[0].set_xlabel("Dice Score")
        axes[0].set_ylabel("Frequencia")
        axes[0].grid(axis="y", alpha=0.3)
        axes[0].legend()
    else:
        axes[0].text(
            0.5,
            0.5,
            f"Sem dados de {subset_label.lower()} para Mid Res",
            ha="center",
            va="center",
            transform=axes[0].transAxes,
        )
        axes[0].set_title(f"Distribuicao dos Dice Scores - Mid Res ({subset_label})")

    if df_high is not None and not df_high.empty:
        # Repete o mesmo fluxo para High.
        dice_scores_high = pd.to_numeric(
            df_high["dice_artery"], errors="coerce"
        ).dropna()
        mu_high = dice_scores_high.mean()
        sigma_high = dice_scores_high.std()
        sns.histplot(
            dice_scores_high,
            bins=20,
            kde=True,
            color="lightgreen",
            edgecolor="black",
            ax=axes[1],
        )
        axes[1].axvline(
            mu_high,
            color="red",
            linestyle="-",
            linewidth=2,
            label=f"Media = {mu_high:.3f}",
        )
        axes[1].axvline(
            mu_high - sigma_high,
            color="orange",
            linestyle="--",
            linewidth=2,
            label=f"-1sigma = {mu_high - sigma_high:.3f}",
        )
        axes[1].axvline(
            mu_high + sigma_high,
            color="orange",
            linestyle="--",
            linewidth=2,
            label=f"+1sigma = {mu_high + sigma_high:.3f}",
        )
        axes[1].axvspan(
            mu_high - sigma_high, mu_high + sigma_high, color="orange", alpha=0.15
        )
        axes[1].set_title(f"Distribuicao dos Dice Scores - High Res ({subset_label})")
        axes[1].set_xlabel("Dice Score")
        axes[1].set_ylabel("Frequencia")
        axes[1].grid(axis="y", alpha=0.3)
        axes[1].legend()
    else:
        axes[1].text(
            0.5,
            0.5,
            f"Sem dados de {subset_label.lower()} para High Res\n(Em preparacao)",
            ha="center",
            va="center",
            transform=axes[1].transAxes,
            fontsize=11,
            color="gray",
        )
        axes[1].set_title(f"Distribuicao dos Dice Scores - High Res ({subset_label})")
        axes[1].set_xticks([])
        axes[1].set_yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd

from utils.project.runtime.notebook_env import get_default_split_paths
from utils.comparison_utils import filter_correct_ostia_cases, load_split_results
from utils.comparison_utils.segmentation_eda import (
    build_success_status_summary_by_subset,
)

SUCCESS_STATUS = ["both_tolerable", "both_correct"]
BAD_OSTIA_STATUS = ["óstios não encontrados", "óstios ruins (bloqueado)"]

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Dados
Os dados de saída do pipeline são carregados a partir do arquivo CSV gerado, contendo as informações de status e coordenadas dos óstios para cada imagem.

In [ ]:
SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
VALID_SPLITS = ("train", "val", "test")
RESOLUTION_KEYS = {"high": "high_res", "mid": "mid_res"}

# Carregar todos os dados diretamente pelo helper compartilhado.
data = {
    resolution: {
        split: load_split_results(
            SPLIT_PATHS_BY_RESOLUTION,
            resolution_key,
            split,
        )
        for split in VALID_SPLITS
    }
    for resolution, resolution_key in RESOLUTION_KEYS.items()
}

# Dataset atualmente ativo (padrão: mid/train)
current_resolution = "mid"
current_split = "train"
df = data[current_resolution][current_split]

available_pairs = [
    f"{resolution}/{split}"
    for resolution, split_data in data.items()
    for split, split_df in split_data.items()
    if split_df is not None
]
print("Dados carregados com sucesso!")
print("\nResolução e split disponíveis:", ", ".join(available_pairs))
print(f"Dados ativos: {current_resolution}/{current_split}")
df.head()

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Distribuição dos Status de Segmentação

### Treino (Train)

In [ ]:
plot_status_distribution_by_subset(data, "train")

### Validação (Val)

In [ ]:
plot_status_distribution_by_subset(data, "val")

### Teste (Test)

In [ ]:
plot_status_distribution_by_subset(data, "test")

## Acertos e Erros na Segmentação

### Treino (Train)

In [ ]:
plot_success_error_by_subset(data, "train", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "train", SUCCESS_STATUS)

### Validação (Val)

In [ ]:
plot_success_error_by_subset(data, "val", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "val", SUCCESS_STATUS)

### Teste (Test)

In [ ]:
plot_success_error_by_subset(data, "test", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "test", SUCCESS_STATUS)

## Distribuição das Distâncias Left e Right (mm)

A linha tracejada vermelha em **7 mm** representa o limite de tolerância adotado para considerar a detecção do óstio como aceitável.

### Treino (Train)

In [ ]:
plot_distance_distribution_by_subset(data, "train", bins=25)

### Validação (Val)

In [ ]:
plot_distance_distribution_by_subset(data, "val", bins=15)

### Teste (Test)

In [ ]:
plot_distance_distribution_by_subset(data, "test", bins=25)

## Distribuição do Dice Score

### Treino (Train)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["train"], data["high"]["train"], "Treino - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["train"]),
    filter_correct_ostia_cases(data["high"]["train"]),
    "Treino - Sem óstios incorretos",
)

### Validação (Val)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["val"], data["high"]["val"], "Validação - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["val"]),
    filter_correct_ostia_cases(data["high"]["val"]),
    "Validação - Sem óstios incorretos",
)

### Teste (Test)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["test"], data["high"]["test"], "Teste - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["test"]),
    filter_correct_ostia_cases(data["high"]["test"]),
    "Teste - Sem óstios incorretos",
)

## Figuras de Dice em mid resolution para publicação
As celulas abaixo geram figuras em alta resolucao separando os casos com sucesso de ostios e os casos com ostios incorretos.


In [ ]:
TEMP_FIGURE_DPI = 400
TEMP_DICE_SPLIT = "test"


def build_mid_dice_subset(
    only_correct_ostia: bool,
    split: str = TEMP_DICE_SPLIT,
) -> pd.DataFrame:
    subset = data["mid"][split].copy()
    if only_correct_ostia:
        subset = filter_correct_ostia_cases(subset)
    subset["dice_artery"] = pd.to_numeric(subset["dice_artery"], errors="coerce")
    return subset.dropna(subset=["dice_artery"])

In [ ]:
mid_dice_all_cases_df = build_mid_dice_subset(only_correct_ostia=False)
_, all_cases_stats = plot_dice_distribution_for_publication(
    mid_dice_all_cases_df["dice_artery"],
    output_path=None,
    dpi=TEMP_FIGURE_DPI,
)
print(all_cases_stats)

In [ ]:
mid_dice_correct_ostia_df = build_mid_dice_subset(only_correct_ostia=True)
_, correct_ostia_stats = plot_dice_distribution_for_publication(
    mid_dice_correct_ostia_df["dice_artery"],
    output_path=None,
    dpi=TEMP_FIGURE_DPI,
)
print(correct_ostia_stats)

## Conclusão

Os resumos de status, distância e Dice permitem avaliar os splits disponíveis e separar o impacto da detecção dos óstios sobre a segmentação arterial.